# Module 2 - Choosing a Model

In this notebook you run two AnyCompany tasks on two different models, measure quality, accuracy, latency and cost, and choose the right model for each task.

## Contents

1. [Introduction](#intro)
2. [Setup](#setup)
3. [Three kinds of model ID](#model-ids)
4. [One function for every model](#helper)
5. [Task 1: travel chat](#chat)
6. [Task 2: expense checks](#expense)
7. [Compare cost](#cost)
8. [Choose a model per task](#choose)
9. [Try it](#try-it)
10. [Conclusion and key takeaways](#conclusion)
11. [Next steps](#next-steps)

## 1. Introduction <a id="intro"></a>

In Module 1 you built a travel chat assistant on Claude Sonnet 5.5. Now AnyCompany's finance team has a request too: check every expense report against the expense rules before it reaches a manager.

The two tasks need different things from a model:

| Task | What matters most |
|:--|:--|
| Travel chat | Helpful answers that follow instructions, including admitting what the assistant does not know |
| Expense checks | Correct verdicts at high volume, for as little cost as possible |

Amazon Bedrock offers models from many providers behind the same Converse API, so you can test several and pick the best one for each task, without rewriting your code. In this module you compare two:

| Model | Provider | Type |
|:--|:--|:--|
| Claude Sonnet 5.5 | Anthropic | Large, high-quality model |
| gpt-oss-120b | OpenAI | Open-weight model with a much lower price per token |

## 2. Setup <a id="setup"></a>

Run the cell below to create the clients and set the two model IDs. The next section explains the different ID formats.

In [ ]:
import json
import re

import boto3

# The AWS Region from your AWS configuration (or the AWS_REGION environment variable).
REGION = boto3.session.Session().region_name

bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)
bedrock = boto3.client("bedrock", region_name=REGION)

MODELS = {
    # Global cross-Region inference profile: requests can run in any AWS Region.
    "Claude Sonnet 5.5": "global.anthropic.claude-sonnet-5-5",
    # In-Region model ID: requests run only in the Region you call.
    "gpt-oss-120b": "openai.gpt-oss-120b-1:0",
}

print(f"Region: {REGION}")
for name, model_id in MODELS.items():
    print(f"  {name:<18} {model_id}")

## 3. Three kinds of model ID <a id="model-ids"></a>

A model ID tells Amazon Bedrock which model to run, and also **where** it may run:

| Format | Example | Where requests run |
|:--|:--|:--|
| In-Region model ID | `openai.gpt-oss-120b-1:0` | Only the Region you call |
| Geographic inference profile | `us.anthropic.claude-sonnet-5-5` | Any Region in that geography (here, the US) |
| Global inference profile | `global.anthropic.claude-sonnet-5-5` | Any commercial AWS Region |

Cross-Region [inference profiles](https://docs.aws.amazon.com/bedrock/latest/userguide/cross-region-inference.html) spread your requests across Regions, which gives you more capacity during busy periods. Choose a geographic profile when your data must stay within a geography. Global profiles are usually slightly cheaper than geographic ones. Some models offer only some of these formats; each [model card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-cards.html) lists them.

Run the cell below to see which Regions the US profile for Claude Sonnet 5.5 can route to.

In [ ]:
profile = bedrock.get_inference_profile(inferenceProfileIdentifier="us.anthropic.claude-sonnet-5-5")

print(profile["inferenceProfileName"])
for model in profile["models"]:
    print("  ", model["modelArn"].split(":")[3])

## 4. One function for every model <a id="helper"></a>

Because Converse uses the same request format for every model, one function can call both. The `ask` function below sends a prompt and returns the answer along with the numbers you need to compare models: latency, input tokens and output tokens.

`get_text` is the helper from Module 1. It skips the reasoning blocks that models can return before their answer.

In [ ]:
def get_text(response):
    """Return the text blocks of a Converse response, joined together."""
    blocks = response["output"]["message"]["content"]
    return "".join(block["text"] for block in blocks if "text" in block)


def ask(model_id, system_prompt, prompt, max_tokens=2000):
    """Send one prompt to a model and return the answer with its usage numbers."""
    response = bedrock_runtime.converse(
        modelId=model_id,
        system=[{"text": system_prompt}],
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": max_tokens},
    )
    return {
        "text": get_text(response).strip(),
        "latency_ms": response["metrics"]["latencyMs"],
        "input_tokens": response["usage"]["inputTokens"],
        "output_tokens": response["usage"]["outputTokens"],
    }

## 5. Task 1: travel chat <a id="chat"></a>

Start with the task from Module 1. Run the cell below to send the same travel question, with the same system prompt, to both models. Read the answers: which would you rather receive as an AnyCompany employee?

In [ ]:
CHAT_SYSTEM_PROMPT = (
    "You are the travel assistant for AnyCompany, helping employees plan and manage "
    "business travel. Be friendly and concise: answer in three sentences or fewer unless "
    "the employee asks for detail. If you are not sure about an AnyCompany-specific rule, "
    "say so and suggest contacting the travel team."
)

CHAT_QUESTION = (
    "My hotel in Chicago is overbooked and has no room for me tonight. "
    "I have a client meeting at 9am tomorrow. What should I do right now?"
)

chat_results = {}
for name, model_id in MODELS.items():
    result = ask(model_id, CHAT_SYSTEM_PROMPT, CHAT_QUESTION)
    chat_results[name] = result
    print(f"=== {name} ({result['latency_ms']} ms, {result['output_tokens']} output tokens)")
    print(result["text"], "\n")

Look closely at what each answer says about AnyCompany. The system prompt tells the assistant to admit when it is not sure about an AnyCompany rule. Claude Sonnet 5.5 does that. gpt-oss-120b usually invents AnyCompany details instead, such as a travel portal or a list of approved hotels, which do not exist. An employee who trusts that answer goes looking for a portal that is not there.

For an assistant that employees rely on, following instructions like this matters more than price.

> **Note:** Judging chat answers is subjective. For a real application, collect a set of representative questions and have people rate the answers, or use [Amazon Bedrock model evaluation](https://docs.aws.amazon.com/bedrock/latest/userguide/evaluation.html) to score them at scale.

## 6. Task 2: expense checks <a id="expense"></a>

The finance team's task is different: every report needs a verdict, `APPROVED` or `FLAGGED`, based on these rules:

| Expense | Rule |
|:--|:--|
| Hotels | Up to $250 per night |
| Meals | Up to $75 per day |
| Flights | Economy for flights under 6 hours; business class allowed for 6 hours or more |

Unlike chat, this task has right and wrong answers, so you can measure accuracy. The file `data/expense_reports.json` holds seven sample reports, each with the verdict a person on the finance team gave it. Run the cell below to load them and look at one.

In [ ]:
with open("../data/expense_reports.json") as f:
    EXPENSE_REPORTS = json.load(f)


def format_report(report):
    """Turn an expense report into the text the model reads."""
    lines = [f"Expense report {report['id']} - {report['employee']} - {report['trip']}"]
    for item in report["items"]:
        lines.append(f"- {item['type']}: {item['description']}: ${item['amount']:.2f}")
    return "\n".join(lines)


print(f"{len(EXPENSE_REPORTS)} reports loaded. The first one:\n")
print(format_report(EXPENSE_REPORTS[0]))
print("\nExpected verdict:", EXPENSE_REPORTS[0]["expected"])

The expense rules go into the system prompt, along with the exact format the reply must follow, so the verdict is easy to read in code. For now, the rules are written into the prompt by hand. In Module 3 you replace them with AnyCompany's actual policy documents.

Run the cell below to check every report with both models, and count the correct verdicts. It takes about a minute.

In [ ]:
EXPENSE_SYSTEM_PROMPT = """You check AnyCompany expense reports against the expense rules below.
Reply with exactly two lines: first line APPROVED or FLAGGED, second line a one-sentence reason.

AnyCompany expense rules:
- Hotels: up to $250 per night.
- Meals: up to $75 per day.
- Flights: economy class for flights under 6 hours; business class allowed for 6 hours or more."""


def get_verdict(text):
    """Find the first APPROVED or FLAGGED in a reply."""
    match = re.search(r"\b(APPROVED|FLAGGED)\b", text)
    return match.group(1) if match else "UNKNOWN"


expense_results = {}
for name, model_id in MODELS.items():
    results = []
    for report in EXPENSE_REPORTS:
        result = ask(model_id, EXPENSE_SYSTEM_PROMPT, format_report(report))
        result["verdict"] = get_verdict(result["text"])
        result["correct"] = result["verdict"] == report["expected"]
        results.append(result)
    expense_results[name] = results
    correct = sum(r["correct"] for r in results)
    print(f"{name:<18} {correct}/{len(results)} correct")

Run the cell below to see each model's verdict next to the finance team's.

In [ ]:
print(f"{'Report':<9} {'Expected':<10} {'Claude Sonnet 5.5':<19} {'gpt-oss-120b':<13}")
for i, report in enumerate(EXPENSE_REPORTS):
    sonnet = expense_results["Claude Sonnet 5.5"][i]["verdict"]
    gpt_oss = expense_results["gpt-oss-120b"][i]["verdict"]
    print(f"{report['id']:<9} {report['expected']:<10} {sonnet:<19} {gpt_oss:<13}")

Both models should get every verdict right. That is not a given: the rules are per night and per day, but the reports list totals, so the model has to divide before it compares. For example, $460 for two nights is $230 a night, within the limit. Both models reason through this step before answering. In testing, faster models that answer without reasoning got several of these reports wrong.

> **Important:** You cannot tell from a model's description whether it handles your task. Always test candidate models on examples of your own task, with known correct answers.

## 7. Compare cost <a id="cost"></a>

On Amazon Bedrock you pay per token, with separate prices for input and output tokens. Each model's price is on the [Amazon Bedrock pricing page](https://aws.amazon.com/bedrock/pricing/).

The cell below uses the Standard tier prices for these model IDs in `us-west-2`, in US dollars per million tokens, as of October 2026. Check the pricing page for your Region and for current prices.

In [ ]:
# USD per 1 million tokens, Standard tier, us-west-2, as of October 2026.
PRICES = {
    "Claude Sonnet 5.5": {"input": 2.00, "output": 10.00},
    "gpt-oss-120b": {"input": 0.15, "output": 0.60},
}


def cost_per_1000(name, results):
    """Average cost of 1,000 requests, based on the token counts in results."""
    input_tokens = sum(r["input_tokens"] for r in results) / len(results)
    output_tokens = sum(r["output_tokens"] for r in results) / len(results)
    price = PRICES[name]
    per_request = (input_tokens * price["input"] + output_tokens * price["output"]) / 1_000_000
    return per_request * 1000


print(f"{'Model':<18} {'Chat, per 1,000':>16} {'Expense check, per 1,000':>26}")
for name in MODELS:
    chat_cost = cost_per_1000(name, [chat_results[name]])
    expense_cost = cost_per_1000(name, expense_results[name])
    print(f"{name:<18} {'$' + format(chat_cost, '.2f'):>16} {'$' + format(expense_cost, '.2f'):>26}")

> **Note:** Both models produce extra output tokens while they reason, and output tokens cost more than input tokens. Always compare the actual cost per request, not only the price per token.

## 8. Choose a model per task <a id="choose"></a>

Run the cell below to put everything side by side.

In [ ]:
print(f"{'Model':<18} {'Chat latency':>13} {'Expense accuracy':>17} {'Expense latency':>16} {'Expense cost, per 1,000':>24}")
for name in MODELS:
    results = expense_results[name]
    correct = sum(r["correct"] for r in results)
    avg_latency = sum(r["latency_ms"] for r in results) / len(results)
    print(
        f"{name:<18} {chat_results[name]['latency_ms']:>10} ms"
        f" {f'{correct}/{len(results)}':>17}"
        f" {avg_latency:>13.0f} ms"
        f" {'$' + format(cost_per_1000(name, results), '.2f'):>24}"
    )

Your numbers will vary a little from run to run, but the pattern should be clear:

- **Travel chat: Claude Sonnet 5.5.** Employees read these answers directly, so quality and following instructions matter most. gpt-oss-120b is cheaper, but invents company details. Chat volume is low enough that the higher price per request is acceptable.
- **Expense checks: gpt-oss-120b.** It gets the verdicts right, and costs roughly a tenth as much as Claude Sonnet 5.5 per check, which adds up across thousands of reports.

Run the cell below to record these choices. You use them for the rest of the workshop.

In [ ]:
CHAT_MODEL_ID = MODELS["Claude Sonnet 5.5"]
EXPENSE_MODEL_ID = MODELS["gpt-oss-120b"]

print("Chat model:         ", CHAT_MODEL_ID)
print("Expense check model:", EXPENSE_MODEL_ID)

## 9. Try it <a id="try-it"></a>

1. **Make Claude Sonnet 5.5 faster and cheaper.** In Module 1 you saw that Claude Sonnet 5.5 decides how much to reason, and that you can set its effort. Run the cell below to check the expense reports again with low effort, then compare accuracy, latency and cost with your results above. Is it enough to change your choice?
2. **Add your own report.** Add a report to `EXPENSE_REPORTS` that tests a rule in a new way, for example a 6-hour flight in business class, and run the checks again.

In [ ]:
low_effort_results = []
for report in EXPENSE_REPORTS:
    response = bedrock_runtime.converse(
        modelId=MODELS["Claude Sonnet 5.5"],
        system=[{"text": EXPENSE_SYSTEM_PROMPT}],
        messages=[{"role": "user", "content": [{"text": format_report(report)}]}],
        inferenceConfig={"maxTokens": 2000},
        # Model-specific setting, passed through to Claude Sonnet 5.5 unchanged.
        additionalModelRequestFields={"output_config": {"effort": "low"}},
    )
    low_effort_results.append({
        "correct": get_verdict(get_text(response)) == report["expected"],
        "latency_ms": response["metrics"]["latencyMs"],
        "input_tokens": response["usage"]["inputTokens"],
        "output_tokens": response["usage"]["outputTokens"],
    })

correct = sum(r["correct"] for r in low_effort_results)
avg_latency = sum(r["latency_ms"] for r in low_effort_results) / len(low_effort_results)
print(f"Claude Sonnet 5.5, low effort: {correct}/{len(low_effort_results)} correct, "
      f"{avg_latency:.0f} ms on average, "
      f"${cost_per_1000('Claude Sonnet 5.5', low_effort_results):.2f} per 1,000 checks")

## 10. Conclusion and key takeaways <a id="conclusion"></a>

- The **Converse API** lets you test models from different providers with the same code; only the model ID changes.
- A **model ID** also decides where requests run: in one Region, a geography, or globally.
- **Measure on your own task.** Quality, accuracy, latency and cost per request depend on the task: gpt-oss-120b is the better choice for expense checks, but not for chat.
- **Reasoning** helps with multi-step work such as arithmetic, at the cost of more output tokens and time.
- Different tasks in one application can, and often should, use **different models**.

## 11. Next steps <a id="next-steps"></a>

Both tasks now have the right model, but the expense rules are written into the prompt by hand, and the chat assistant still knows nothing about AnyCompany's travel policy. In [Module 3 - Grounding with a Managed Knowledge Base](03_knowledge_base.ipynb), you teach both to answer from AnyCompany's actual policy documents.